# W04-3 입면 분할과 LOD1 매스

<a href="https://colab.research.google.com/github/kloud80/dataminig_cau/blob/main/%EC%A4%91%EC%95%99%EB%8C%8026%ED%8A%B9%EA%B0%95/notebooks/w04_colab/W04_3%20%EC%9E%85%EB%A9%B4%20%EB%B6%84%ED%95%A0%EA%B3%BC%20LOD1%20%EB%A7%A4%EC%8A%A4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

바닥 윤곽(1번)과 높이(2번)가 모였다. 이제 **3D** 다.
윤곽을 정리하고, 벽 하나하나(**입면**)로 나누고, 높이만큼 밀어 올린다. 이것이 **LOD1** — 지붕이 평평한 상자 모델이다.

여기까지 **사진도, 인공지능도 쓰지 않는다.** 전부 결정론적인 기하 계산이다. 같은 입력이면 항상 같은 결과가 나온다.
파이프라인은 이 입면 목록 위에서 비로소 거리 사진을 찍고(어느 벽을 몇 장), 판독한다(창 몇 개, 층 경계 어디).

| 단계 | 하는 일 | 파이프라인 스테이지 |
|---|---|---|
| 1 | 윤곽 단순화 — 일직선 꼭짓점·짧은 변 정리 | `s04_geometry` · `geometry/footprint.simplify` |
| 2 | 변 하나 = 입면 하나: 폭·바깥 방향·향(방위) | `footprint.build_facades` |
| 3 | 맞벽 판정 — 옆 건물에 붙은 벽은 사진을 못 찍는다 | `build_facades` (party_wall) |
| 4 | 실제 실행(s04)의 입면 표와 대조 | `s04_geometry.json` |
| 5 | 높이만큼 밀어 올려 LOD1 매스 → `.glb` | `procedural.lod1_mesh` |

> 결과물: `out/facades.json`, `out/lod1.glb`, `out/w04_3_facades.png`, `out/w04_3_lod1.png`

## 1. 준비

In [ ]:
# ── 0. 실습 데이터 확인 ─────────────────────────────────────────
# 이 노트북은 옆에 data/ipark/ 폴더가 있다고 가정한다 (아이파크타워 실제 파이프라인 결과).
import os, sys, json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/kloud80/dataminig_cau.git"   # 강의 저장소 (필요한 폴더만 받는다)
REPO_DIR = "중앙대26특강/notebooks/w04_colab"

DATA = Path('data/ipark')
if not DATA.exists() and REPO_URL:
    print('실습 데이터를 내려받는 중...')
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--filter=blob:none', '--sparse', REPO_URL, 'w04_repo'], check=False)
    subprocess.run(['git', '-C', 'w04_repo', 'sparse-checkout', 'set', REPO_DIR], check=False)
    hits = sorted(Path('w04_repo').glob('**/data/ipark'))
    if hits:
        os.chdir(hits[0].parent.parent)        # data/ 의 부모 폴더로 이동
if not DATA.exists():
    raise FileNotFoundError(
        'data/ipark/ 폴더가 없습니다.\n'
        '  1) 수업 저장소를 clone 하고 이 노트북을 w04_colab 폴더에서 여세요.\n'
        '  2) Colab 이라면 네트워크를 확인하고 이 셀을 다시 실행하세요.\n'
        '  (또는 data/ipark 폴더를 왼쪽 파일 탭에 그대로 올려도 됩니다)')

OUT = Path('out'); OUT.mkdir(exist_ok=True)
print('작업 폴더', Path.cwd())
print('실습 데이터', sorted(p.name for p in DATA.iterdir()))

In [ ]:
try:
    import trimesh, mapbox_earcut           # trimesh 의 다각형 밀어올리기는 삼각분할기(earcut)가 필요하다
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'trimesh', 'mapbox_earcut'], check=True)
    import trimesh
import math
from shapely.geometry import Polygon, LineString
from shapely.ops import unary_union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 한글 폰트 — 그래프 글자가 네모로 깨지지 않게 (Colab 은 나눔고딕을 설치, 윈도는 맑은 고딕)
import platform
import matplotlib.font_manager as fm
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        os.system('apt-get -qq install -y fonts-nanum > /dev/null')
    if os.path.exists(nanum):
        fm.fontManager.addfont(nanum)
        plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

## 2. 입력 — 윤곽과 높이

윤곽은 두 가지 중에서 고른다.

* `'gis'` — 실제 실행에서 쓴 **국가 GIS 윤곽** (실습 데이터 `s03_prior.json`). **기본값.** 실제 결과(s04)와 1:1 로 비교할 수 있다.
* `'osm'` — 1번 노트북에서 받은 **OSM 윤곽** (`out/footprint.json`). 폴백으로 만들면 무엇이 달라지는지 본다.

높이는 2번 노트북의 `out/prior.json` 이 있으면 그것을, 없으면 실습 데이터를 쓴다.

In [ ]:
FOOTPRINT_SOURCE = 'gis'      # 'gis' 또는 'osm'

s03 = json.load(open(DATA / 's03_prior.json', encoding='utf-8'))
s04 = json.load(open(DATA / 's04_geometry.json', encoding='utf-8'))

if FOOTPRINT_SOURCE == 'osm' and (OUT / 'footprint.json').exists():
    fpj = json.load(open(OUT / 'footprint.json', encoding='utf-8'))
    raw_fp, origin = fpj['footprint_local'], fpj['origin_lnglat']
    neighbors = fpj['neighbors']
    fp_source = f"osm ({fpj['source']})"
else:
    if FOOTPRINT_SOURCE == 'osm':
        print('out/footprint.json 이 없다 (1번 노트북을 먼저 돌릴 것) → gis 로 진행')
    raw_fp, origin = s03['footprint_local'], s03['origin_lnglat']
    neighbors = s03['neighbors']
    fp_source = 'gis (s03 실습 데이터)'

if (OUT / 'prior.json').exists():
    pj = json.load(open(OUT / 'prior.json', encoding='utf-8'))
    prior_src = 'out/prior.json'
else:
    pj, prior_src = s03, 's03 실습 데이터'
HEIGHT = pj['height_m']
FLOOR_HEIGHTS = pj['floor_heights']

print(f'윤곽  {fp_source} — 꼭짓점 {len(raw_fp)}개, 면적 {Polygon(raw_fp).area:,.1f} ㎡')
print(f'높이  {HEIGHT} m · {len(FLOOR_HEIGHTS)}층  ({prior_src})')
print(f'주변 건물 {len(neighbors)}동')

## 3. 윤곽 단순화

국가 GIS 윤곽에는 수십 cm 짜리 꺾임이 많다. 입면 하나하나가 사진 촬영·판독·텍스처의 **작업 단위**이므로,
쓸모없는 꺾임은 일을 늘리기만 한다. 파이프라인의 규칙은 두 가지다.

1. **거의 일직선인 꼭짓점** (꺾인 각도 6° 미만) → 지운다
2. **짧은 변** (0.5 m 미만) → 두 끝점 중 지웠을 때 면적이 덜 변하는 쪽을 지운다

먼저 꼭짓점을 **반시계방향(CCW)** 으로 맞춘다. 그래야 "변의 오른쪽 = 건물 밖" 이 항상 성립한다.

In [ ]:
def signed_area(r):
    x, y = r[:, 0], r[:, 1]
    return 0.5 * float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def to_ccw(ring):
    a = np.asarray(ring, float)
    if len(a) > 1 and np.allclose(a[0], a[-1]):
        a = a[:-1]
    return a if signed_area(a) > 0 else a[::-1].copy()

def turn_deg(prev, cur, nxt):
    v1, v2 = cur - prev, nxt - cur
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-9 or n2 < 1e-9:
        return 0.0
    return math.degrees(math.acos(np.clip(v1 @ v2 / (n1 * n2), -1, 1)))

def simplify(ring, min_edge_m=0.5, collinear_deg=6.0):
    """파이프라인 geometry/footprint.simplify 와 같은 규칙."""
    a = to_ccw(ring)
    for _ in range(500):
        n = len(a)
        if n <= 4:
            break
        turns = np.array([turn_deg(a[i - 1], a[i], a[(i + 1) % n]) for i in range(n)])
        i = int(np.argmin(turns))
        if turns[i] < collinear_deg:                    # 1) 일직선 꼭짓점
            a = np.delete(a, i, axis=0); continue
        edges = np.linalg.norm(np.roll(a, -1, axis=0) - a, axis=1)
        j = int(np.argmin(edges))
        if edges[j] < min_edge_m:                       # 2) 짧은 변
            base = abs(signed_area(a))
            k1, k2 = j, (j + 1) % n
            d1 = abs(abs(signed_area(np.delete(a, k1, axis=0))) - base)
            d2 = abs(abs(signed_area(np.delete(a, k2, axis=0))) - base)
            a = np.delete(a, k1 if d1 <= d2 else k2, axis=0); continue
        break
    return to_ccw(a)

def iou(a, b):
    pa, pb = Polygon(a).buffer(0), Polygon(b).buffer(0)
    return pa.intersection(pb).area / pa.union(pb).area

ring0 = to_ccw(raw_fp)
fp = simplify(ring0)
print(f'꼭짓점 {len(ring0)} → {len(fp)} · IoU {iou(ring0, fp):.4f}')
print('변 길이(m):', np.round(np.linalg.norm(np.roll(fp, -1, axis=0) - fp, axis=1), 2))

### 다른 방법 — shapely 의 Douglas–Peucker
GIS 에서 흔히 쓰는 `simplify(tolerance)` 는 **원래 선에서 tolerance 이상 벗어나지 않는** 범위에서 점을 지운다.
허용 오차를 키우면 입면 수가 줄지만 모양이 망가진다. 얼마가 적당한가?

In [ ]:
rows, shapes = [], {}
for tol in [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]:
    s = Polygon(ring0).simplify(tol, preserve_topology=True) if tol else Polygon(ring0)
    r = np.asarray(s.exterior.coords)[:-1]
    shapes[tol] = r
    rows.append({'허용 오차(m)': tol, '꼭짓점=입면 수': len(r), '면적(㎡)': round(s.area, 1),
                 '면적 변화(%)': round((s.area / Polygon(ring0).area - 1) * 100, 2), 'IoU': round(iou(ring0, r), 4)})
display(pd.DataFrame(rows).set_index('허용 오차(m)'))

fig, ax = plt.subplots(1, 3, figsize=(15, 6))
for a, (title, r) in zip(ax, [('원본', ring0), ('파이프라인 규칙', fp), ('shapely 2.0 m', shapes[2.0])]):
    a.fill(*np.vstack([r, r[:1]]).T, fc='#F4C7BE', ec='#B0281A', lw=2)
    a.plot(*r.T, 'o', color='#B0281A', ms=5)
    a.set_title(f'{title} — 꼭짓점 {len(r)}개', fontsize=13); a.set_aspect('equal'); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

아이파크타워의 국가 GIS 윤곽은 이미 깔끔해서 **파이프라인 규칙으로는 하나도 지워지지 않는다** (가장 짧은 변이 0.52 m 로 기준 0.5 m 를 겨우 넘는다).
그 0.52 m 짜리 변도 입면 하나가 된다. 이런 좁은 입면은 사진을 따로 찍지 않고 이웃 입면의 텍스처를 빌려 쓴다 (`narrow`).

## 4. 입면 나누기 — 변 하나 = 벽 하나

CCW 윤곽의 변 A→B 하나가 입면 하나다. **밖에서 보면 A 가 왼쪽, B 가 오른쪽**이다.

* **폭** = |B − A|
* **바깥 법선** = 변 방향을 오른쪽으로 90° 돌린 것: `(eᵧ, −eₓ) / 폭` — CCW 이므로 오른쪽이 건물 밖이다
* **향(向)** = 법선의 방위각 (북 0°, 시계방향). 남향 벽 = 법선이 남쪽을 보는 벽
* **narrow** = 폭 2.5 m 미만 → 사진 대신 이웃 입면 텍스처를 빌린다
* **캔버스** = 텍스처 해상도. 3 cm 당 1픽셀, 긴 변 최대 4,096 px

In [ ]:
TEXEL_M, CANVAS_MAX, MIN_TEX_W, PARTY_GAP = 0.03, 4096, 2.5, 0.3     # 파이프라인 설정값
COMPASS = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
KOR = {'N': '북', 'NE': '북동', 'E': '동', 'SE': '남동', 'S': '남', 'SW': '남서', 'W': '서', 'NW': '북서'}

def bearing_deg(dx, dy):
    """ENU 벡터의 방위각 (북 0°, 시계방향)."""
    return math.degrees(math.atan2(dx, dy)) % 360

def compass(b):
    return COMPASS[int((b + 22.5) // 45) % 8]

def build_facades(ring, height):
    out = []
    n = len(ring)
    for i in range(n):
        A, B = ring[i], ring[(i + 1) % n]
        e = B - A
        w = float(np.linalg.norm(e))
        if w < 1e-6:
            continue
        normal = np.array([e[1], -e[0]]) / w
        cw, ch = max(16, round(w / TEXEL_M)), max(16, round(height / TEXEL_M))
        k = min(1.0, CANVAS_MAX / max(cw, ch))
        b = bearing_deg(*normal)
        out.append({'id': f'f_{i}', 'start': A.tolist(), 'end': B.tolist(), 'width_m': round(w, 3),
                    'height_m': height, 'normal': normal.round(4).tolist(), 'bearing_deg': round(b, 1),
                    'compass': compass(b), 'narrow': w < MIN_TEX_W,
                    'canvas_px': [max(16, int(cw * k)), max(16, int(ch * k))]})
    return out

facades = build_facades(fp, HEIGHT)
df = pd.DataFrame(facades)[['id', 'width_m', 'bearing_deg', 'compass', 'narrow', 'canvas_px']]
df['향'] = df['compass'].map(KOR)
display(df)
print(f'입면 {len(facades)}개 · 둘레 {df.width_m.sum():.1f} m · 가장 넓은 입면 {df.width_m.max():.1f} m ({df.loc[df.width_m.idxmax(), "향"]}향)')

## 5. 맞벽 판정 — 옆 건물에 붙은 벽

옆 건물에 딱 붙은 벽은 거리에서 보이지 않는다. 사진을 찍을 수도, 창을 그릴 수도 없다.
파이프라인의 판정은 이렇다.

1. 주변 건물을 전부 합쳐 **0.3 m 부풀린다** (맞벽 구역)
2. 입면 선을 **바깥으로 0.15 m 밀어낸다**
3. 밀어낸 선의 **60% 넘게** 맞벽 구역 안에 들어가면 → 맞벽

그런데 **주변 건물 목록이 틀리면** 판정도 틀린다. 1번 노트북에서 본 사고 — OSM 의 아이파크타워 자신이 주변 건물로 남은 것 — 를 다시 꺼낸다.
두 가지 주변 건물 목록으로 판정해 본다.

* **A. 파이프라인 그대로** — 대상과 50% 넘게 겹치는 것만 뺀 목록 (실제 실행)
* **B. 고친 규칙** — 대상과 **조금이라도**(1 ㎡ 넘게) 겹치면 같은 건물로 보고 뺀다

In [ ]:
def party_flags(ring, facades, nb_rings):
    polys = [Polygon(p).buffer(0) for p in nb_rings if len(p) >= 3]
    if not polys:
        return [False] * len(facades)
    zone = unary_union(polys).buffer(PARTY_GAP)
    flags = []
    for f in facades:
        A, B, nrm = np.array(f['start']), np.array(f['end']), np.array(f['normal'])
        shifted = LineString([A + nrm * PARTY_GAP * 0.5, B + nrm * PARTY_GAP * 0.5])
        flags.append(shifted.intersection(zone).length / max(f['width_m'], 1e-6) > 0.6)
    return flags

T = Polygon(fp)
nb_A = [n['footprint_local'] for n in neighbors]
overlap = {n['building_id']: Polygon(n['footprint_local']).buffer(0).intersection(T).area for n in neighbors}
nb_B = [n['footprint_local'] for n in neighbors if overlap[n['building_id']] <= 1.0]
dropped = [k for k, v in overlap.items() if v > 1.0]
print('B 규칙으로 빠진 "주변 건물":', [(k, f'{overlap[k]:.0f} ㎡ 겹침') for k in dropped] or '없음')

flags_A, flags_B = party_flags(fp, facades, nb_A), party_flags(fp, facades, nb_B)
df['맞벽 A (파이프라인)'] = flags_A
df['맞벽 B (고친 규칙)'] = flags_B
display(df[['id', 'width_m', '향', 'narrow', '맞벽 A (파이프라인)', '맞벽 B (고친 규칙)']])
for f, b in zip(facades, flags_B):
    f['party_wall'] = b                     # 저장에는 고친 규칙을 쓴다

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 9))
lim = 60
for a, flags, title, nbs in [(ax[0], flags_A, 'A. 파이프라인 그대로', nb_A), (ax[1], flags_B, 'B. 겹친 "이웃" 을 뺀 뒤', nb_B)]:
    for r in nbs:
        r = np.asarray(r)
        a.fill(*np.vstack([r, r[:1]]).T, fc='#C9D3E3', ec='#6B7A90', lw=0.8, alpha=0.7)
    for d in dropped:
        if nbs is nb_A:
            r = np.asarray(next(n['footprint_local'] for n in neighbors if n['building_id'] == d))
            a.fill(*np.vstack([r, r[:1]]).T, fc='none', ec='#7B2CBF', lw=2.5, ls='--')
            a.text(*r.mean(0), '같은 건물의\nOSM 윤곽', color='#7B2CBF', fontsize=11, ha='center', weight='bold')
    a.fill(*np.vstack([fp, fp[:1]]).T, fc='#FBE3DE', ec='none')
    for f, p in zip(facades, flags):
        A, B, nrm = np.array(f['start']), np.array(f['end']), np.array(f['normal'])
        a.plot(*np.array([A, B]).T, color='k' if p else '#E8452C', lw=5 if p else 3, ls='-' if not p else (0, (2, 1)))
        M = (A + B) / 2
        if f['width_m'] >= 4:
            a.annotate('', xy=M + nrm * 7, xytext=M, arrowprops=dict(arrowstyle='->', color='#B0281A', lw=1.5))
            a.text(*(M + nrm * 11), f"{f['id']}\n{KOR[f['compass']]}", ha='center', va='center', fontsize=10)
    a.set_xlim(-lim, lim); a.set_ylim(-lim, lim); a.set_aspect('equal'); a.grid(alpha=0.3)
    a.set_title(f'{title} — 맞벽 {sum(flags)}개', fontsize=14)
    a.set_xlabel('동쪽 (m)'); a.set_ylabel('북쪽 (m)')
fig.suptitle('입면 분할 — 빨강: 촬영 대상 입면 · 검정 점선: 맞벽 · 화살표: 바깥 법선(향)', fontsize=14)
plt.tight_layout(); plt.savefig(OUT / 'w04_3_facades.png', dpi=150); plt.show()

**A** 에서는 동쪽의 짧은 벽 몇 개가 맞벽으로 잡힌다. 그런데 그 자리에 붙어 있다는 "옆 건물" 은 **같은 건물을 OSM 이 다르게 그린 것**이다.
맞벽으로 잡힌 입면은 촬영 계획에서 빠지고, 창도 그리지 않는다. 데이터 한 줄의 실수가 3D 모델의 벽 하나를 **빈 벽**으로 만든다.

> 규칙 B 도 완벽하지 않다. 실제로 **붙어 지은 두 건물**(상가 연립)의 윤곽이 1 ㎡ 넘게 겹쳐 그려져 있으면, 진짜 이웃을 지워 버린다.
> 규칙을 고칠 때는 **무엇을 새로 망가뜨리는지**도 같이 본다.

## 6. 실제 실행(s04)과 대조

실습 데이터 `s04_geometry.json` 은 파이프라인이 실제로 만든 입면 표다. 같은 윤곽(`gis`)이면 폭·법선·좁은 입면이 **똑같아야** 한다.
맞벽은 규칙 A 와 같아야 한다.

In [ ]:
def compare_by_direction():
    # 윤곽이 다르면 f_0, f_1 ... 번호가 가리키는 벽이 다르다 → 향별 폭 합계로만 비교한다
    def by_dir(fs):
        s = pd.Series({KOR[c]: 0.0 for c in COMPASS})
        for f in fs:
            s[KOR[compass(bearing_deg(*f['normal']))]] += f['width_m']
        return s.round(1)
    display(pd.DataFrame({'우리 (OSM)': by_dir(facades), 's04 (국가 GIS)': by_dir(s04['facades'])}).T)
    print(f"입면 수: 우리 {len(facades)} / s04 {len(s04['facades'])}  — 윤곽이 달라 입면 번호로 맞대지 않는다")

def compare_by_id():
    ref = {f['id']: f for f in s04['facades']}
    rows = []
    for f, pa in zip(facades, flags_A):
        r = ref.get(f['id'])
        if r is None:
            rows.append({'id': f['id'], 's04 에 있나': False}); continue
        dang = abs((bearing_deg(*f['normal']) - bearing_deg(*r['normal']) + 180) % 360 - 180)
        rows.append({'id': f['id'], 's04 에 있나': True, '폭 차이(mm)': round(abs(f['width_m'] - r['width_m']) * 1000, 2),
                     '법선 차이(°)': round(dang, 3), 'narrow 같음': f['narrow'] == r['narrow'],
                     '맞벽 s04': r['party_wall'], '맞벽 A': pa, '맞벽 B': f['party_wall'],
                     '캔버스 같음': f['canvas_px'] == r['canvas_px']})
    cmp = pd.DataFrame(rows)
    display(cmp)
    ok = cmp['s04 에 있나'].all() and len(facades) == len(s04['facades'])
    print(f"입면 수: 우리 {len(facades)} / s04 {len(s04['facades'])}")
    if ok:
        print('폭·법선·narrow·캔버스 모두 일치:', bool((cmp['폭 차이(mm)'] < 1).all() and (cmp['법선 차이(°)'] < 0.01).all()
              and cmp['narrow 같음'].all() and cmp['캔버스 같음'].all()))
        print('맞벽 A == s04:', bool((cmp['맞벽 A'] == cmp['맞벽 s04']).all()),
              '/ 맞벽 B 와 다른 입면:', list(cmp.loc[cmp['맞벽 B'] != cmp['맞벽 s04'], 'id']))

if fp_source.startswith('gis'):
    compare_by_id()
else:
    compare_by_direction()
print('\n(s04 에는 street_facing 도 있다 — 거리 사진이 그 벽을 볼 수 있는지. 로드뷰 위치가 필요해 여기서는 다루지 않는다.)')

## 7. LOD1 — 높이만큼 밀어 올린다

윤곽을 높이만큼 위로 밀면(extrude) 바닥·지붕·벽으로 닫힌 상자가 된다.
3D 모델이 **물 샐 틈 없이 닫혀 있는지**(watertight)와 **면이 바깥을 보는지**(부피가 +)를 꼭 확인한다 — 파이프라인은 이 검사에 실패하면 멈춘다.

그다음 **glTF(.glb)** 로 저장한다. glTF 는 **Y 가 위**다. 우리 좌표는 **Z 가 위**(ENU)이므로 축을 돌려서 쓴다: `(x, y, z) → (x, z, −y)`.
벽은 **향**별로 색을 칠해 둔다.

In [ ]:
mesh = trimesh.creation.extrude_polygon(Polygon(fp), HEIGHT)
print(f'꼭짓점 {len(mesh.vertices)} · 삼각형 {len(mesh.faces)}')
print(f'닫혀 있나(watertight) {mesh.is_watertight} · 면 방향 일관 {mesh.is_winding_consistent} · 부피 {mesh.volume:,.0f} ㎥')
print(f'바닥면적 × 높이 = {T.area:,.1f} × {HEIGHT} = {T.area * HEIGHT:,.0f} ㎥')

# 실제 실행의 LOD1 (OBJ, 이미 Y-up) 과 비교
obj = trimesh.load(DATA / 'mesh_lod1.obj', force='mesh', process=False)
v = np.asarray(obj.vertices)
obj_enu = np.column_stack([v[:, 0], -v[:, 2], v[:, 1]])           # (x, z, -y) 를 되돌림
ours = np.unique(np.round(mesh.vertices, 3), axis=0)
theirs = np.unique(np.round(obj_enu, 3), axis=0)
if ours.shape == theirs.shape:
    print(f's04 mesh_lod1.obj 와 꼭짓점 최대 차이 {np.abs(ours - theirs).max() * 1000:.2f} mm')
else:
    print(f's04 mesh_lod1.obj 꼭짓점 {len(theirs)}개 vs 우리 {len(ours)}개 (윤곽 출처가 다르면 당연히 다르다)')

In [ ]:
PALETTE = {'N': '#4E79A7', 'NE': '#76B7B2', 'E': '#59A14F', 'SE': '#EDC948',
           'S': '#F28E2B', 'SW': '#E15759', 'W': '#B07AA1', 'NW': '#9C755F'}
def hex_rgba(h):
    return [int(h[i:i + 2], 16) for i in (1, 3, 5)] + [255]

m = mesh.copy()
m.unmerge_vertices()                       # 면마다 색을 따로 주려면 꼭짓점을 나눈다
fc = []
for nrm in m.face_normals:
    if nrm[2] > 0.9:
        fc.append([200, 200, 200, 255])    # 지붕
    elif nrm[2] < -0.9:
        fc.append([90, 90, 90, 255])       # 바닥
    else:
        fc.append(hex_rgba(PALETTE[compass(bearing_deg(nrm[0], nrm[1]))]))
m.visual.face_colors = np.array(fc, dtype=np.uint8)

Z_UP_TO_Y_UP = np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, -1, 0, 0], [0, 0, 0, 1]], float)
m.apply_transform(Z_UP_TO_Y_UP)
scene = trimesh.Scene()
scene.add_geometry(m, node_name='lod1', geom_name='lod1')
scene.metadata.update({'crs': 'LOCAL_ENU (glTF Y-up: x=동, y=위, -z=북)', 'origin_lnglat': origin,
                       'height_m': HEIGHT, 'footprint_source': fp_source})
glb = OUT / 'lod1.glb'
glb.write_bytes(scene.export(file_type='glb'))
print('저장', glb, f'{glb.stat().st_size / 1024:.1f} KB')

## 8. 3D 로 보기

마우스로 돌려 보자. (Colab·Jupyter 에서 동작한다. 인터넷에서 뷰어 스크립트를 받아 온다.)

In [ ]:
import base64
from IPython.display import HTML
b64 = base64.b64encode(glb.read_bytes()).decode()
display(HTML(f"""
<script type="module" src="https://unpkg.com/@google/model-viewer@3.5.0/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{b64}" camera-controls auto-rotate
  camera-orbit="30deg 70deg auto" shadow-intensity="1" exposure="1.1"
  style="width:100%;height:520px;background:#F3F4F6"></model-viewer>
<p style="font-size:13px">벽 색 = 향 · {' · '.join(f'<span style="color:{c}">■</span> {KOR[k]}' for k, c in PALETTE.items())}</p>
"""))

### 주변 건물과 함께 (trimesh 뷰어)
반경 80 m 안의 주변 건물을 OSM 높이(없으면 9 m)로 세운다. 주변이 대부분 **9 m 기본값**이라는 것이 한눈에 보인다 — 실제로는 더 높은 건물도 있다.

In [ ]:
ctx = trimesh.Scene()
ctx.add_geometry(m, node_name='target')
for n, r in zip(neighbors, nb_A):
    if n['building_id'] in dropped:
        continue
    P = Polygon(r).buffer(0)
    if P.is_empty or P.geom_type != 'Polygon' or P.distance(T) > 80:
        continue
    nm = trimesh.creation.extrude_polygon(P, n['height_m'])
    nm.visual.face_colors = [185, 192, 205, 255] if n['height_source'] != 'default' else [225, 225, 225, 255]
    nm.apply_transform(Z_UP_TO_Y_UP)
    ctx.add_geometry(nm, node_name=n['building_id'])
try:
    from trimesh.viewer import notebook
    display(notebook.scene_to_notebook(ctx, height=520))
except Exception as e:
    print('뷰어를 띄우지 못함:', e)

### 슬라이드용 정지 그림

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')
z_lines = np.cumsum([0] + list(FLOOR_HEIGHTS))
for f in facades:
    A, B = np.array(f['start']), np.array(f['end'])
    col = '#3A3A3A' if f['party_wall'] else PALETTE[f['compass']]
    floors = [[[*A, z0], [*B, z0], [*B, z1], [*A, z1]] for z0, z1 in zip(z_lines[:-1], z_lines[1:])]
    ax.add_collection3d(Poly3DCollection(floors, fc=col, ec='#FFFFFF', lw=0.4, alpha=0.95))   # 층마다 한 장
ax.add_collection3d(Poly3DCollection([[[*p, HEIGHT] for p in fp]], fc='#D0D0D0', ec='k', lw=0.8))
for n, r in zip(neighbors, nb_A):
    if n['building_id'] in dropped:
        continue
    r = np.asarray(r); P = Polygon(r).buffer(0)
    if P.distance(T) > 45:
        continue
    h = n['height_m']
    for i in range(len(r)):
        a, b = r[i], r[(i + 1) % len(r)]
        ax.add_collection3d(Poly3DCollection([[[*a, 0], [*b, 0], [*b, h], [*a, h]]], fc='#C9CED6', ec='#9AA3AF', lw=0.3, alpha=0.35))
L = 55
ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_zlim(0, 2 * L)
ax.set_box_aspect((1, 1, 1)); ax.view_init(elev=24, azim=-125)
ax.set_xlabel('동쪽 (m)'); ax.set_ylabel('북쪽 (m)'); ax.set_zlabel('높이 (m)')
ax.set_title(f'LOD1 매스 — 높이 {HEIGHT} m · {len(FLOOR_HEIGHTS)}층 · 입면 {len(facades)}개 (색 = 향, 짙은 회색 = 맞벽)', fontsize=13)
plt.tight_layout(); plt.savefig(OUT / 'w04_3_lod1.png', dpi=150); plt.show()

## 9. 저장 — `out/facades.json`, `out/lod1.glb`

In [ ]:
out = {'crs': 'LOCAL_ENU', 'origin_lnglat': origin, 'footprint_source': fp_source,
       'footprint': fp.round(3).tolist(), 'height_m': HEIGHT, 'floor_heights': FLOOR_HEIGHTS,
       'simplify': {'vertices_before': len(ring0), 'vertices_after': len(fp), 'iou': round(iou(ring0, fp), 4)},
       'party_rule': 'B: 대상과 1㎡ 넘게 겹치는 주변 건물은 같은 건물로 보고 제외',
       'facades': facades}
json.dump(out, open(OUT / 'facades.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

by = df.groupby('향')['width_m'].sum().sort_values(ascending=False)
print('── W04-3 요약 ───────────────────────────')
print(f'윤곽       {fp_source} — 꼭짓점 {len(ring0)} → {len(fp)} (단순화 후)')
print(f'바닥면적   {T.area:,.1f} ㎡ · 높이 {HEIGHT} m · 부피 {mesh.volume:,.0f} ㎥')
print(f'입면       {len(facades)}개 · 좁은 입면 {sum(f["narrow"] for f in facades)}개 · '
      f'맞벽 {sum(flags_B)}개 (파이프라인 규칙으로는 {sum(flags_A)}개)')
print('향별 입면 폭', ', '.join(f'{k} {v:.1f} m' for k, v in by.items()))
print(f'LOD1       watertight={mesh.is_watertight} · {glb.name} {glb.stat().st_size / 1024:.1f} KB')
print('저장      ', ', '.join(sorted(p.name for p in OUT.glob('*'))))

## 해 볼 것

1. 2번 셀의 `FOOTPRINT_SOURCE` 를 `'osm'` 으로 바꿔 다시 돌려 보자. 입면 수와 향별 폭이 어떻게 달라지는가? 그 모델로 사진 촬영 계획을 세우면 무엇이 틀어질까?
2. 3번의 `simplify` 기준을 `min_edge_m=2.0` 으로 올려 보자. 입면이 몇 개로 줄고 IoU 는 얼마인가? 작업량과 정확도 중 무엇을 택하겠는가?
3. 아이파크타워의 가장 넓은 입면은 어느 향인가? 위성사진(1번 노트북 7번 그림)에서 그 벽이 어느 도로를 보는지 확인해 보자.
4. 맞벽 규칙 B 가 **진짜 이웃을 잘못 지우는** 경우를 하나 생각해 보자. 그것까지 막으려면 어떤 정보가 더 필요할까? (힌트: 같은 PNU, 같은 이름, 겹친 비율)
5. LOD1 은 지붕이 평평하고 층이 다 같다. 다음 단계(LOD2·LOD3)로 가려면 무엇이 더 필요한가 — 그것은 공공데이터에 있는가, 사진에서 읽어야 하는가?

### 참고
* addr2blend 파이프라인 (사내 연구용) — `geometry/footprint.py` (simplify, build_facades), `stages/s04_geometry.py`
* OGC CityGML 3.0 — 건물 상세수준(LOD 0~3) 정의
* glTF 2.0 (Khronos) — Y-up, 미터 단위 · `<model-viewer>` (Google) · trimesh